In [5]:
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

# Change these for each recording.
DATA_ROOT = Path.home() / "markerless" / "markerless_sam" / "data" / "opensim"
SUBJECT = "S10"
TRIAL = "lunge_0_1"

subject_dir = DATA_ROOT / SUBJECT

MIN_FORCE_N = 20.0
MIN_OVERLAP_S = 0.10
UP_AXIS = "y"  # OpenSim ground frame used by the S10 exports

# If both versions exist, use the filtered one first.
PREFER_FILTERED = True

def pick_file(*names):
    for name in names:
        path = subject_dir / name
        if path.is_file():
            return path
    raise FileNotFoundError(
        f"No matching file in {subject_dir}. Tried: {', '.join(names)}"
    )

def pick_trial_file(filtered, unfiltered):
    order = (filtered, unfiltered) if PREFER_FILTERED else (unfiltered, filtered)
    return pick_file(*order)

subject_number = int(SUBJECT.removeprefix("S"))
MODEL_PATH = pick_file(f"Subj_{subject_number}_Hu2025.osim")
IK_PATH = pick_trial_file(f"{TRIAL}_ik_filt.mot", f"{TRIAL}_ik.mot")
EXTERNAL_LOADS_PATH = pick_trial_file(
    f"{TRIAL}_extload_filt.xml",
    f"{TRIAL}_extload.xml",
)

external_loads = ET.parse(EXTERNAL_LOADS_PATH).getroot()
datafile = external_loads.findtext(".//datafile", "").strip()
if not datafile:
    raise ValueError(f"No <datafile> in {EXTERNAL_LOADS_PATH.name}")

# Keep the filename from the XML, even if it contains an old Windows path.
grf_name = datafile.replace("\\", "/").rsplit("/", 1)[-1]
GRF_PATH = pick_file(grf_name)

for label, path in [
    ("Model", MODEL_PATH),
    ("IK", IK_PATH),
    ("External loads", EXTERNAL_LOADS_PATH),
    ("GRF", GRF_PATH),
]:
    print(f"{label}: {path.name}")


def read_mot(path):
    with path.open(encoding="utf-8-sig") as file:
        for line in file:
            if line.strip().lower() == "endheader":
                data = pd.read_csv(file, sep=r"\s+")
                break
        else:
            raise ValueError(f"No endheader in {path.name}")

    time = data["time"].to_numpy(dtype=float)
    if len(time) < 2 or not np.isfinite(time).all() or np.any(np.diff(time) <= 0):
        raise ValueError(f"Expected increasing, finite timestamps in {path.name}")
    return data


ik = read_mot(IK_PATH)
grf = read_mot(GRF_PATH)
external_loads = ET.parse(EXTERNAL_LOADS_PATH).getroot()

# Use the XML's force identifiers and foot-body assignments.
# For models with different foot names, update this mapping.
foot_side = {"calcn_l": "left", "calcn_r": "right"}
force_columns = {"left": [], "right": []}

for load in external_loads.findall(".//ExternalForce"):
    body = load.findtext("applied_to_body", "").strip()
    if body not in foot_side:
        continue

    frame = load.findtext("force_expressed_in_body", "").strip()
    if frame != "ground":
        raise ValueError(f"Expected ground-frame force for {load.get('name')}")

    prefix = load.findtext("force_identifier", "").strip()
    column = f"{prefix}{UP_AXIS}"
    if not prefix or column not in grf.columns:
        raise ValueError(f"Missing vertical force column in {GRF_PATH.name}: {column}")

    force_columns[foot_side[body]].append(column)

if not all(force_columns.values()):
    raise ValueError(
        f"Expected left and right foot loads in {EXTERNAL_LOADS_PATH.name}. "
        "Check foot_side against the model's body names."
    )

time = grf["time"].to_numpy()
dt = np.median(np.diff(time))
in_ik_range = (time >= ik["time"].iloc[0]) & (time <= ik["time"].iloc[-1])

left_loaded = grf[force_columns["left"]].gt(MIN_FORCE_N).any(axis=1)
right_loaded = grf[force_columns["right"]].gt(MIN_FORCE_N).any(axis=1)
both_loaded = (left_loaded & right_loaded).to_numpy() & in_ik_range

# Split on missing samples as well as changes in contact.
indices = np.flatnonzero(both_loaded)
breaks = np.flatnonzero(
    (np.diff(indices) != 1) | (np.diff(time[indices]) > 1.5 * dt)
) + 1

intervals = []
for segment in np.split(indices, breaks):
    if segment.size == 0:
        continue
    start, end = time[segment[[0, -1]]]
    duration = end - start + dt
    if duration >= MIN_OVERLAP_S:
        intervals.append((start, end, duration, len(segment)))

candidate_windows = pd.DataFrame(
    intervals, columns=["start_s", "end_s", "duration_s", "samples"]
)

print(f"{SUBJECT} / {TRIAL}")
print("Vertical force columns:", force_columns)
display(candidate_windows.round(3))

Model: Subj_10_Hu2025.osim
IK: lunge_0_1_ik.mot
External loads: lunge_0_1_extload_filt.xml
GRF: lunge_0_1_filt.mot
S10 / lunge_0_1
Vertical force columns: {'left': ['ground_force3_vy', 'ground_force1_vy'], 'right': ['ground_force2_vy']}


,start_s,end_s,duration_s,samples
0,7.068,8.003,0.936,936
1,8.507,9.591,1.085,1085


In [ ]:
# check which Vicon motion and scaled OpenSim model files are available for that window.
import sys
import opensim as osim

print(sys.executable, osim.__file__)

if candidate_windows.empty:
    raise ValueError(f"No bilateral overlap found for {SUBJECT}/{TRIAL}")

# First candidate window for a diagnostic run.
window = candidate_windows.iloc[0]
ik_times = ik.loc[
    ik["time"].between(window["start_s"], window["end_s"]), "time"
]
if len(ik_times) < 2:
    raise ValueError("The selected window contains fewer than two IK frames.")

start_s, end_s = float(ik_times.iloc[0]), float(ik_times.iloc[-1])

repo_root = DATA_ROOT.resolve().parents[1]
out_dir = repo_root / "results" / SUBJECT / TRIAL / "inverse_dynamics"
out_dir.mkdir(parents=True, exist_ok=True)

# Replace the Windows path in a copy of the external-load XML.
loads_tree = ET.parse(EXTERNAL_LOADS_PATH)
loads_tree.find(".//datafile").text = str(GRF_PATH.resolve())
local_loads = out_dir / "external_loads_local.xml"
loads_tree.write(local_loads, encoding="utf-8", xml_declaration=True)

# Save the settings so this run can be reproduced.
setup = ET.Element("OpenSimDocument", Version=loads_tree.getroot().get("Version", "40000"))
tool = ET.SubElement(setup, "InverseDynamicsTool", name=f"{SUBJECT}_{TRIAL}")
output_name = "joint_moments.sto"

settings = {
    "model_file": str(MODEL_PATH.resolve()),
    "coordinates_file": str(IK_PATH.resolve()),
    "external_loads_file": str(local_loads),
    "results_directory": str(out_dir),
    "time_range": f"{start_s:.8f} {end_s:.8f}",
    "forces_to_exclude": "Muscles",
    "lowpass_cutoff_frequency_for_coordinates": "-1",
    "output_gen_force_file": output_name,
}
for name, value in settings.items():
    ET.SubElement(tool, name).text = value

setup_path = out_dir / "setup_inverse_dynamics.xml"
ET.ElementTree(setup).write(setup_path, encoding="utf-8", xml_declaration=True)

print(f"Running {SUBJECT}/{TRIAL} from {start_s:.3f} to {end_s:.3f} s")
if not osim.InverseDynamicsTool(str(setup_path)).run():
    raise RuntimeError("OpenSim inverse dynamics failed. Check the OpenSim log.")

print(f"Joint moments: {out_dir / output_name}")

ModuleNotFoundError: No module named 'opensim'